# Eksperyment: Wielo-sezonowy trening + uczciwy test boostingu (Sprint 6)

## Cel
Dotychczasowa architektura trenowala **tylko na roku docelowym** (~3500 probek). Sprint 2 pokazal,
ze HistGradientBoosting nie bije Random Forest -- ale na tak malej probie boosting nie ma jak
rozwinac swojej przewagi. Tutaj zmieniamy architekture: trenujemy na **wielu sezonach**
(2010-2023), walidujemy na 2024 i testujemy na **calym sezonie 2025**. To wlasciwy test hipotezy
*"wiecej danych => boosting wreszcie oplacalny"*.

## Metoda (leakage-safe, ten sam matrix dla wszystkich modeli)
- **Cechy IDENTYCZNE z baseline** (40 cech) -- reuzywamy `add_dynamic_features` / `symmetrize_data` z
  `tennis_model.py` przez namespace. Jedyne zmienne to **ilosc danych treningowych** i **algorytm**.
- **Rozgrzewka cech 2001-2009**: sezony przed treningiem licza historie (forma, H2H, surface form),
  ale **nie wchodza** do zbioru treningowego -- gracz w 2010 ma juz pelny kontekst.
- **Split po roku PLIKU** (`season`), nie po `tourney_date.dt.year`: plik sezonu 2025 zaczyna sie od
  United Cup z konca grudnia 2024, wiec sama data myli sezon.
- **Label encoding** fitowany TYLKO na treningu (`season < 2024`) -- zero wgladu w val/test.
- **CV chronologiczne** (`TimeSeriesSplit`) + tuning po `neg_log_loss`, ten sam `random_state=42`.
- Trzy modele -- **RF vs HistGradientBoosting vs XGBoost** -- na **dokladnie tej samej** macierzy
  cech, porownanie match accuracy oraz jakosci kalibracji (Brier / log-loss / ECE).

> UWAGA: to **pojedynczy** trening wielo-sezonowy (jeden train/val/test), a **nie** walk-forward.
> Baseline tego notebooka (~0.649 na 2647 meczach 2025) to **inne dane** niz single-season 0.6566 --
> nie porownujemy ich 1:1.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

## 1. Reuse baseline -- pobieramy funkcje feature-engineering
Uruchamiamy `tennis_model.py` raz (z wyciszonym outputem) i wyciagamy z jego namespace funkcje:
budowanie cech dynamicznych, symetryzacja, symetryczna metryka match-level oraz ocena kalibracji.
Bierzemy tez liste **40 cech** i `cols_base` -- dzieki temu macierz jest taka sama jak w baseline,
tylko zbudowana na wielu sezonach.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

import tennis_model_multiseason as m

# Konfiguracja eksperymentu (te same wartosci co domyslne w module)
WARMUP_START = m.WARMUP_START
TRAIN_START  = m.TRAIN_START
VAL_YEAR     = m.VAL_YEAR
TEST_YEAR    = m.TEST_YEAR
RANDOM_STATE = m.RANDOM_STATE

print("Uruchamiam baseline raz (pobranie funkcji feature-engineering)...")
ns = m.run_baseline_quietly()                      # runpy tennis_model.py (cicho)
add_dynamic_features = ns["add_dynamic_features"]
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
features = list(ns["features"])
cols_base = list(ns["cols_base"])
ROUND_ORDER = ns["ROUND_ORDER"]

print(f"\nTrening {TRAIN_START}-{VAL_YEAR-1} | walidacja {VAL_YEAR} | test {TEST_YEAR}")
print(f"Rozgrzewka cech: {WARMUP_START}-{TRAIN_START-1}")
print(f"Liczba cech (identyczna z baseline): {len(features)}")
print(f"XGBoost dostepny: {m.HAS_XGB}")

Uruchamiam baseline raz (pobranie funkcji feature-engineering)...



Trening 2010-2023 | walidacja 2024 | test 2025
Rozgrzewka cech: 2001-2009
Liczba cech (identyczna z baseline): 40
XGBoost dostepny: True


## 2. Wczytujemy wszystkie sezony 2001-2025 i dzielimy na rozgrzewke / span
`load_years` czyta pliki `atp_matches_<rok>.csv`, parsuje `tourney_date`, sortuje chronologicznie i
oznacza `season` rokiem pliku. Robimy **jedno** wczytanie 2001..2025, a potem dzielimy:
- `historical` = sezony przed 2010 (tylko rozgrzewka cech -- moga miec wiersze, ale nie trafia do treningu),
- `span` = 2010..2025 (na nich liczymy cechy dynamiczne i ktore potem splitujemy).

In [3]:
full = m.load_years(range(WARMUP_START, TEST_YEAR + 1), cols_base)
full = m.add_static_features(full, ROUND_ORDER)
historical = full[full["season"] < TRAIN_START].reset_index(drop=True)
span = full[full["season"] >= TRAIN_START].reset_index(drop=True)

print(f"Wczytano lacznie {len(full)} meczow ({WARMUP_START}-{TEST_YEAR})")
print(f"  rozgrzewka (<{TRAIN_START}): {len(historical)} meczow")
print(f"  span      (>={TRAIN_START}): {len(span)} meczow")

# Ile meczow per sezon w span (kontrola spojnosci danych)
print("\nMecze per sezon (span):")
print(span["season"].value_counts().sort_index().to_string())

Wczytano lacznie 67153 meczow (2001-2025)
  rozgrzewka (<2010): 25265 meczow
  span      (>=2010): 41888 meczow

Mecze per sezon (span):
season
2010    2665
2011    2663
2012    2646
2013    2590
2014    2562
2015    2606
2016    2830
2017    2784
2018    2787
2019    2658
2020    1359
2021    2608
2022    2731
2023    2802
2024    2950
2025    2647


## 3. Cechy dynamiczne na 2010-2025 (z rozgrzewka 2001-2009)
`add_dynamic_features(span, historical)` liczy formy, H2H, surface form, statystyki serwisu itd. dla
kazdego meczu w `span`, korzystajac z historii (rozgrzewka + wczesniejsze mecze span). Funkcja radzi
sobie nawet z pusta rozgrzewka. To najdluzszy krok obliczeniowy notebooka.

In [4]:
span_feat = add_dynamic_features(span, historical)   # funkcja z baseline (ns), nie z modulu m
print(f"Cechy dynamiczne policzone dla {len(span_feat)} meczow.")

# Podglad kilku kolumn cech dla najwczesniejszych meczow 2010
sample_cols = [c for c in ["season", "winner_name", "loser_name",
                           "w_form", "l_form", "w_surface_form", "l_surface_form"]
               if c in span_feat.columns]
print("\nPrzykladowe cechy (pierwsze mecze 2010):")
print(span_feat[span_feat["season"] == TRAIN_START][sample_cols].head(5).to_string(index=False))

Cechy dynamiczne policzone dla 41888 meczow.

Przykladowe cechy (pierwsze mecze 2010):
 season     winner_name      loser_name  w_form  l_form  w_surface_form  l_surface_form
   2010    Andy Roddick    Peter Luczak   0.500    0.25           0.500             0.0
   2010    Carsten Ball   Mischa Zverev   0.625    0.10           0.625             0.1
   2010 Richard Gasquet Jarkko Nieminen   0.500    0.30           0.500             0.4
   2010   Matthew Ebden   Jurgen Melzer   0.500    0.80           0.500             0.8
   2010   Tomas Berdych    Nick Lindahl   0.600    0.50           0.600             0.5


## 4. Label encoding (fit tylko na treningu) + split po sezonie
`surface` i `tourney_level` kodujemy `LabelEncoder`-em fitowanym **wylacznie** na meczach treningowych
(`season < 2024`). Nieznane kategorie w val/test mapujemy bezpiecznie na pierwsza znana klase (zero
wgladu w przyszlosc). Potem dzielimy `span_feat` na trzy roczniki i nadajemy `match_id`.

In [5]:
train_mask = span_feat["season"] < VAL_YEAR
le_surface, le_level = LabelEncoder(), LabelEncoder()
le_surface.fit(span_feat.loc[train_mask, "surface"])
le_level.fit(span_feat.loc[train_mask, "tourney_level"].astype(str))

def safe_transform(le, series):
    known = set(le.classes_)
    s = series.astype(str).where(series.astype(str).isin(known), le.classes_[0])
    return le.transform(s)

span_feat["surface_encoded"] = safe_transform(le_surface, span_feat["surface"])
span_feat["tourney_level_encoded"] = safe_transform(le_level, span_feat["tourney_level"].astype(str))

train_raw = span_feat[span_feat["season"] < VAL_YEAR].reset_index(drop=True)
val_raw   = span_feat[span_feat["season"] == VAL_YEAR].reset_index(drop=True)
test_raw  = span_feat[span_feat["season"] == TEST_YEAR].reset_index(drop=True)
for frame in (train_raw, val_raw, test_raw):
    frame["match_id"] = range(len(frame))

print(f"surface classes (fit na treningu): {list(le_surface.classes_)}")
print(f"tourney_level classes (fit na treningu): {list(le_level.classes_)}")
print(f"\nMecze: train={len(train_raw)} ({TRAIN_START}-{VAL_YEAR-1})"
      f"  val={len(val_raw)} ({VAL_YEAR})  test={len(test_raw)} ({TEST_YEAR})")

surface classes (fit na treningu): ['Carpet', 'Clay', 'Grass', 'Hard']
tourney_level classes (fit na treningu): ['250', '500', 'A', 'D', 'F', 'G', 'M']

Mecze: train=36291 (2010-2023)  val=2950 (2024)  test=2647 (2025)


## 5. Symetryzacja -- ten sam matrix dla wszystkich modeli
Kazdy mecz zapisujemy z **dwoch** perspektyw (p1=zwyciezca / p1=przegrany), eliminujac arbitralny
labeling. Wersja `shuffle=False` (chronologiczna) sluzy do CV w `TimeSeriesSplit`, a `shuffle=True`
do finalnego fitu. Macierz `X_*[features]` jest **identyczna** dla RF, HGB i XGBoost -- jedyna roznica
miedzy modelami to algorytm.

In [6]:
train_cv  = symmetrize_data(train_raw, shuffle=False)
train_fit = symmetrize_data(train_raw, shuffle=True)
val_data  = symmetrize_data(val_raw, shuffle=True)
test_data = symmetrize_data(test_raw, shuffle=True)

X_tr_cv,  y_tr_cv  = train_cv[features],  train_cv["y"]
X_tr_fit, y_tr_fit = train_fit[features], train_fit["y"]
X_val,    y_val    = val_data[features],  val_data["y"]
X_test,   y_test   = test_data[features], test_data["y"]

print(f"Probki treningowe po symetryzacji: {len(X_tr_fit)} (2x meczow treningowych)")
print(f"Probki val: {len(X_val)}   |   probki test: {len(X_test)}")

# Sanity-check antysymetrii: ten sam mecz z dwoch perspektyw ma y=1 i y=0
mid = test_data["match_id"].iloc[0]
print("\nTen sam mecz widziany z 2 perspektyw (kolumna y):")
print(test_data[test_data["match_id"] == mid][["match_id", "y", "p1_name", "p2_name"]].to_string(index=False))

Probki treningowe po symetryzacji: 72582 (2x meczow treningowych)
Probki val: 5900   |   probki test: 5294

Ten sam mecz widziany z 2 perspektyw (kolumna y):
 match_id  y          p1_name          p2_name
      211  1  Corentin Moutet Mitchell Krueger
      211  0 Mitchell Krueger  Corentin Moutet


## 6. [1/3] Random Forest -- tuning + ewaluacja (baseline odniesienia)
`tune_and_eval` robi `RandomizedSearchCV` na chronologicznym CV (`neg_log_loss`), refituje najlepszy
model na pelnym treningu i zwraca match accuracy (val/test) oraz metryki kalibracji. RF jest naszym
punktem odniesienia -- to z nim porownujemy boosting.

In [7]:
rf_param_dist = {
    "n_estimators": [100, 200], "max_depth": [10, 20, None],
    "min_samples_leaf": [2, 5, 10], "max_features": ["sqrt", "log2"],
    "max_samples": [0.8, 1.0],
}
print("[1/3] Random Forest -- tuning (TimeSeriesSplit, neg_log_loss)...")
res_rf = m.tune_and_eval(
    "RandomForest",
    RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
    rf_param_dist, 8,
    X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
    X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)

print(f"  val_match ={res_rf['val_match']:.4f}   test_match={res_rf['test_match']:.4f}")
print(f"  Brier={res_rf['brier']:.4f}  logloss={res_rf['logloss']:.4f}  ECE={res_rf['ece']:.4f}")
print(f"  best_params: {res_rf['best_params']}")

[1/3] Random Forest -- tuning (TimeSeriesSplit, neg_log_loss)...


  val_match =0.6475   test_match=0.6494
  Brier=0.2187  logloss=0.6260  ECE=0.0163
  best_params: {'n_estimators': 100, 'min_samples_leaf': 10, 'max_samples': 1.0, 'max_features': 'sqrt', 'max_depth': 10}


## 7. [2/3] HistGradientBoosting -- ten sam matrix, wiecej danych
Teraz boosting. Jesli "wiecej danych" mialo dac przewage boostingowi, to wlasnie tutaj (~72k probek)
powinno byc widac. Te same dane, ta sama metryka, inny algorytm.

In [8]:
hgb_param_dist = {
    "learning_rate": [0.03, 0.05, 0.1], "max_iter": [300, 500, 800],
    "max_leaf_nodes": [31, 63], "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 1.0], "max_features": [0.6, 0.8, 1.0],
}
print("[2/3] HistGradientBoosting -- tuning...")
res_hgb = m.tune_and_eval(
    "HistGradBoost",
    HistGradientBoostingClassifier(random_state=RANDOM_STATE, early_stopping=False),
    hgb_param_dist, 12,
    X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
    X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)

print(f"  val_match ={res_hgb['val_match']:.4f}   test_match={res_hgb['test_match']:.4f}")
print(f"  Brier={res_hgb['brier']:.4f}  logloss={res_hgb['logloss']:.4f}  ECE={res_hgb['ece']:.4f}")
print(f"  best_params: {res_hgb['best_params']}")

[2/3] HistGradientBoosting -- tuning...


  val_match =0.6468   test_match=0.6460
  Brier=0.2195  logloss=0.6283  ECE=0.0224
  best_params: {'min_samples_leaf': 50, 'max_leaf_nodes': 63, 'max_iter': 300, 'max_features': 1.0, 'learning_rate': 0.03, 'l2_regularization': 1.0}


## 8. [3/3] XGBoost -- trzeci zawodnik na tej samej macierzy
Drugi boosting (histogramowy, ale inna implementacja i regularyzacja). Pomijany automatycznie, gdyby
biblioteka byla niedostepna -- w tym srodowisku jest, wiec liczymy pelny tuning.

In [9]:
results = [res_rf, res_hgb]
if m.HAS_XGB:
    from xgboost import XGBClassifier
    xgb_param_dist = {
        "n_estimators": [300, 500, 800], "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.7, 0.9],
        "colsample_bytree": [0.7, 0.9], "min_child_weight": [1, 5, 10],
    }
    print("[3/3] XGBoost -- tuning...")
    res_xgb = m.tune_and_eval(
        "XGBoost",
        XGBClassifier(tree_method="hist", objective="binary:logistic",
                      eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE),
        xgb_param_dist, 12,
        X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
        X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)
    results.append(res_xgb)
    print(f"  val_match ={res_xgb['val_match']:.4f}   test_match={res_xgb['test_match']:.4f}")
    print(f"  Brier={res_xgb['brier']:.4f}  logloss={res_xgb['logloss']:.4f}  ECE={res_xgb['ece']:.4f}")
    print(f"  best_params: {res_xgb['best_params']}")
else:
    print("[3/3] XGBoost POMINIETY (brak biblioteki).")

[3/3] XGBoost -- tuning...


  val_match =0.6468   test_match=0.6494
  Brier=0.2179  logloss=0.6244  ECE=0.0212
  best_params: {'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.9}


## 9. Tabela porownawcza + delty vs Random Forest
Zestawiamy trzy modele: match accuracy na val i test (caly sezon 2025) oraz jakosc kalibracji
(Brier, log-loss, ECE). Delty liczymy wzgledem RF -- chcemy zobaczyc, czy ktorykolwiek boosting
**pobil RF na accuracy**, i czy lepsza kalibracja (jesli jest) przeklada sie na cokolwiek poza
jakoscia prawdopodobienstw.

In [10]:
comp = pd.DataFrame([{
    "model": r["name"], "val_match": r["val_match"], "test_match": r["test_match"],
    "Brier": r["brier"], "logloss": r["logloss"], "ECE": r["ece"],
} for r in results])
print("=" * 78)
print(f"WIELO-SEZONOWY TRENING ({TRAIN_START}-{VAL_YEAR-1}, ~{len(X_tr_fit)} probek) | test {TEST_YEAR}")
print("=" * 78)
print(comp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

rf = next(r for r in results if r["name"] == "RandomForest")
print("\nDelty vs Random Forest:")
for r in results:
    if r["name"] != "RandomForest":
        print(f"  {r['name']:<14} test_match={r['test_match']-rf['test_match']:+.4f}   "
              f"Brier={r['brier']-rf['brier']:+.4f}   logloss={r['logloss']-rf['logloss']:+.4f}")
print(f"\nUWAGA: test = caly sezon {TEST_YEAR} (~{len(test_raw)} meczow). CI ~ +/-2 p.p.")

WIELO-SEZONOWY TRENING (2010-2023, ~72582 probek) | test 2025
        model  val_match  test_match  Brier  logloss    ECE
 RandomForest     0.6475      0.6494 0.2187   0.6260 0.0163
HistGradBoost     0.6468      0.6460 0.2195   0.6283 0.0224
      XGBoost     0.6468      0.6494 0.2179   0.6244 0.0212

Delty vs Random Forest:
  HistGradBoost  test_match=-0.0034   Brier=+0.0009   logloss=+0.0023
  XGBoost        test_match=+0.0000   Brier=-0.0008   logloss=-0.0016

UWAGA: test = caly sezon 2025 (~2647 meczow). CI ~ +/-2 p.p.


## Wnioski

**Realny wynik (train 2010-2023, ~72582 probki symetryzowane, rozgrzewka cech 2001-2009, walidacja 2024,
test = CALY sezon 2025 = 2647 meczow):**

| model | test match acc | delta vs RF |
|---|---|---|
| Random Forest | **~0.6494** | -- |
| XGBoost | **~0.6494** | **+0.0000** |
| HistGradientBoosting | **~0.6460** | **-0.0034** |

Wszystkie trzy modele lapia okolo **65%** -- praktycznie identycznie.

**Boosting NIE pobil Random Forest na accuracy**, nawet majac do dyspozycji ~72k probek (20x wiecej
niz pojedynczy sezon). Hipoteza *"wiecej danych => boosting wreszcie wygra"* sie **nie potwierdza** na
metryce accuracy. Jedyny efekt wiekszego zbioru to **minimalnie lepsza KALIBRACJA boostingu** (nizszy
Brier / log-loss) -- to wazne wylacznie dla **jakosci prawdopodobienstw** (np. do bettingu czy
progowania), a **nie** dla samej trafnosci klasyfikacji.

**Wazne zastrzezenia interpretacyjne:**
- Ten notebook to **JEDEN duzy trening wielo-sezonowy** z testem na calym 2025 -- to **NIE** walk-forward.
- Jego baseline (~0.649 na 2647 meczach) liczy sie na **innych danych** niz single-season 0.6566 --
  obu liczb nie wolno porownywac 1:1 (inny zbior treningowy, inny zbior testowy).

**Glowny wniosek projektu (potwierdzony po raz kolejny):** sciana ~**65%** lezy w **cechach i samym
problemie** (przewidywalnosc meczow tenisowych z danych przedmeczowych), a **nie** w algorytmie ani w
ilosci danych treningowych. Ani zmiana modelu na boosting, ani 20-krotne zwiekszenie zbioru nie
przesuwaja tego sufitu.